### import env var and libraries

In [3]:
from openai import OpenAI
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

load_dotenv()

client = OpenAI()

### set up Pushover

In [5]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


In [6]:
#Test pushover
import requests

def send_notification(message: str):
    payload = { "user": pushover_user, "token": pushover_token, "message": message }
    response = requests.post(pushover_url, data=payload)
    return response

In [8]:
send_notification("Hello from VScode!")

<Response [200]>

### describe pushover as an LLM tool

In [ ]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a pushover notification to the user's phone via the pushover API. Use this to alert the user about important information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
             }
            },
        "required": ["message"]
        }
}

### add pushover to the list of tools for LLM

In [11]:
tools = [{"type": "function", "function": send_notification_function}]

### Calling the tool from an LLM

In [ ]:
from litellm import completion
client = OpenAI()
response = completion.create(
    model="gpt-4.1-mini",
    messages = [{"role": "user", "content": "Send me a fun fact about software engineering someone who is trying to land an intern role would like to hear (context: I am a first year Data Scienc major in SJSU)"}],
    tools=tools,
    tool_choice="auto",
    messages=[
        {"role": "user", "content": "Send me a notification."}
    ]
)